In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [ ]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

In [ ]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

In [ ]:
label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])
data

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder()
geo=ohe.fit_transform(data[['Geography']])    
geo.toarray()

In [ ]:
geo_df=pd.DataFrame(geo.toarray(),columns=ohe.get_feature_names_out(['Geography']))


In [ ]:
data=pd.concat([data.drop('Geography',axis=1),geo_df],axis=1)
data.head()

In [ ]:
with open('label_encoder.pkl','wb')as file:
    pickle.dump(label_encoder,file)

with open('ohe.pkl','wb')as file:
    pickle.dump(ohe,file)


In [ ]:
x=data.drop('Exited',axis=1)
y=data['Exited']  


In [ ]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
with open('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
model=Sequential([Dense(64,activation='relu',input_shape=(X_train.shape[1],)),Dense(32,activation='relu'),Dense(1,activation='sigmoid')])

In [ ]:
model.summary()

In [ ]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
early_stopping=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[tensorflow_callback,early_stopping])

In [ ]:
model.save('model.h5')

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit/20260201-175639